<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/2_BiLSTM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import joblib
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv('BBCA_Master_Dataset_BiLSTM.csv')

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed')
df = df.sort_values('Date')


print(f"Data loaded successfully. Total rows: {len(df)}")
print("\n--- Data Verification ---")
print(df[['Date', 'Close', 'Sentiment_Score']].head(3))

df = df.dropna().reset_index(drop=True)

In [ ]:
features_base = ['Open', 'High', 'Low', 'Close', 'Volume']
features_enh = ['Open', 'High', 'Low', 'Close', 'Volume', 'Sentiment_Score']
target = ['Close']

scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

scaled_all = scaler_features.fit_transform(df[features_enh])
scaled_target = scaler_target.fit_transform(df[target])

joblib.dump(scaler_features, 'scaler_features.pkl')
joblib.dump(scaler_target, 'scaler_target.pkl')

def create_sequences(data, target_data, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(target_data[i + window_size])
    return np.array(X), np.array(y)

# Create sequences
WINDOW_SIZE = 10
X_base, y = create_sequences(scaled_all[:, :5], scaled_target, WINDOW_SIZE)
X_enh, _ = create_sequences(scaled_all, scaled_target, WINDOW_SIZE)

# Split 80/20
split = int(len(X_base) * 0.8)
X_train_base, X_test_base = X_base[:split], X_base[split:]
X_train_enh, X_test_enh = X_enh[:split], X_enh[split:]
y_train, y_test = y[:split], y[split:]

print(f"Sequences created. Training on {len(X_train_base)} days, Testing on {len(X_test_base)} days.")

In [ ]:
def build_bilstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        # Bidirectional layer processes sequences in both directions
        Bidirectional(LSTM(units=64, return_sequences=True)),
        Dropout(0.2),
        # Second LSTM layer to condense information
        LSTM(units=32, return_sequences=False),
        Dropout(0.2),
        Dense(units=16, activation='relu'),
        Dense(units=1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

In [ ]:
# Train Baseline (Numerical Only)
print("Training Baseline Model...")
model_base = build_bilstm((WINDOW_SIZE, 5))
model_base.fit(X_train_base, y_train, validation_data=(X_test_base, y_test),
               epochs=100, batch_size=16, callbacks=[early_stop], verbose=0)

# Train Sentiment-Enhanced (With Lagged Sentiment)
print("Training Sentiment-Enhanced Model...")
model_enh = build_bilstm((WINDOW_SIZE, 6))
model_enh.fit(X_train_enh, y_train, validation_data=(X_test_enh, y_test),
              epochs=100, batch_size=16, callbacks=[early_stop], verbose=0)

model_enh.save('BBCA_BiLSTM_Sentiment_Model.keras')
print("Models trained and Sentiment model saved.")

In [ ]:

pred_base = model_base.predict(X_test_base)
pred_enh = model_enh.predict(X_test_enh)

actual_prices = scaler_target.inverse_transform(y_test).flatten()
prices_base = scaler_target.inverse_transform(pred_base).flatten()
prices_enh = scaler_target.inverse_transform(pred_enh).flatten()

rmse_base = math.sqrt(mean_squared_error(actual_prices, prices_base))
mae_base = mean_absolute_error(actual_prices, prices_base)
mape_base = np.mean(np.abs((actual_prices - prices_base) / actual_prices)) * 100
r2_base = r2_score(actual_prices, prices_base)

rmse_enh = math.sqrt(mean_squared_error(actual_prices, prices_enh))
mae_enh = mean_absolute_error(actual_prices, prices_enh)
mape_enh = np.mean(np.abs((actual_prices - prices_enh) / actual_prices)) * 100
r2_enh = r2_score(actual_prices, prices_enh)

def calculate_win_rate(actual, predicted):
    actual_direction = np.sign(actual[1:] - actual[:-1])
    pred_direction = np.sign(predicted[1:] - actual[:-1])
    correct_directions = np.sum(actual_direction == pred_direction)
    return (correct_directions / len(actual_direction)) * 100

win_rate_base = calculate_win_rate(actual_prices, prices_base)
win_rate_enh = calculate_win_rate(actual_prices, prices_enh)

print("\n" + "="*75)
print(f"{'EVALUATION METRIC':<28} | {'BASELINE (PRICE ONLY)':<20} | {'ENHANCED (+SENTIMENT)':<20}")
print("="*75)
print(f"{'RMSE (Lower=Better)':<28} | Rp {rmse_base:<16.2f} | Rp {rmse_enh:<16.2f}")
print(f"{'MAE (Lower=Better)':<28} | Rp {mae_base:<16.2f} | Rp {mae_enh:<16.2f}")
print(f"{'MAPE (Lower=Better)':<28} | {mape_base:<17.2f}% | {mape_enh:<17.2f}%")
print(f"{'R² Variance Score':<28} | {r2_base:<18.4f} | {r2_enh:<18.4f}")
print(f"{'Directional Win Rate':<28} | {win_rate_base:<17.2f}% | {win_rate_enh:<17.2f}%")
print("="*75 + "\n")

# Creating a custom grid layout to handle 5 individual sections
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1])
fig.suptitle('Thesis Evaluation Dashboard: BBCA Model Performance Comparison', fontsize=16, fontweight='bold', y=0.98)

c_base = '#e63946'
c_enh = '#2a9d8f'

# --- Plot 1: Full Stock History Tracker (Timeline Alignment Patch) ---
ax0 = fig.add_subplot(gs[0, :])

# Convert to 1D flat arrays explicitly to prevent dimensional broadcasting shifts
actual_plot = np.array(actual_prices).flatten()
base_plot = np.array(prices_base).flatten()
enh_plot = np.array(prices_enh).flatten()

# Generate uniform time steps starting from 0 to N-1
time_steps = np.arange(len(actual_plot))

ax0.plot(time_steps, actual_plot, label='Actual BBCA Price', color='#1d3557', linewidth=2.5)
ax0.plot(time_steps, base_plot, label=f'Baseline BiLSTM (RMSE: Rp {rmse_base:.0f})', color=c_base, linestyle='--', alpha=0.85)
ax0.plot(time_steps, enh_plot, label=f'Sentiment-Enhanced BiLSTM (RMSE: Rp {rmse_enh:.0f})', color=c_enh, linestyle='-.', alpha=0.95)

ax0.set_title('Price Prediction Trajectory Tracker', fontsize=12, fontweight='bold')
ax0.set_xlabel('Test Set Days', fontsize=10)
ax0.set_ylabel('Price (IDR)', fontsize=10)
ax0.set_xlim(0, len(actual_plot) - 1)
ax0.legend(loc='upper left')
ax0.grid(True, alpha=0.25)

# --- Plot 2: Absolute Errors Bar Chart (RMSE & MAE) ---
ax1 = fig.add_subplot(gs[1, 0])
labels_err = ['RMSE', 'MAE']
x = np.arange(len(labels_err))
width = 0.35

rects1 = ax1.bar(x - width/2, [rmse_base, mae_base], width, label='Baseline', color=c_base)
rects2 = ax1.bar(x + width/2, [rmse_enh, mae_enh], width, label='Enhanced', color=c_enh)
ax1.set_title('Absolute Error Comparison (Lower is Better)', fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_err)
ax1.set_ylabel('IDR (Rupiah)')
ax1.legend()
ax1.grid(True, alpha=0.15)
ax1.bar_label(rects1, padding=3, fmt='Rp %.1f')
ax1.bar_label(rects2, padding=3, fmt='Rp %.1f')

# --- Plot 3: Isolated MAPE Comparison ---
ax2 = fig.add_subplot(gs[1, 1])
y_pos = np.arange(2)
mape_vals = [mape_base, mape_enh]
rects_mape = ax2.barh(y_pos, mape_vals, height=0.5, color=[c_base, c_enh])
ax2.set_title('Mean Absolute Percentage Error (MAPE - Lower is Better)', fontsize=12, fontweight='bold')
ax2.set_yticks(y_pos)
ax2.set_yticklabels(['Baseline Model', 'Enhanced Model'])
ax2.set_xlabel('Percentage Error (%)')
ax2.grid(True, alpha=0.15)
ax2.bar_label(rects_mape, padding=5, fmt='%.3f%%', fontweight='bold')
ax2.set_xlim(0, max(mape_vals) * 1.2)

# --- Plot 4: R² Variance Score Comparison ---
ax3 = fig.add_subplot(gs[2, 0])
rects3 = ax3.bar(['Baseline', 'Enhanced'], [r2_base, r2_enh], width=0.5, color=[c_base, c_enh])
ax3.set_title('R² Coefficient of Determination (Higher is Better)', fontsize=12, fontweight='bold')
ax3.set_ylabel('R² Value Score')
ax3.grid(True, alpha=0.15)
ax3.bar_label(rects3, padding=3, fmt='%.4f', fontweight='bold')
ax3.set_ylim(0, max(r2_base, r2_enh) * 1.15)

# --- Plot 5: Directional Trading Win Rate ---
ax4 = fig.add_subplot(gs[2, 1])
rects4 = ax4.bar(['Baseline Win %', 'Enhanced Win %'], [win_rate_base, win_rate_enh], width=0.5, color=[c_base, c_enh])
ax4.set_title('Directional Trend Accuracy (Trading Win Rate)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Percentage (%)')
ax4.grid(True, alpha=0.15)
ax4.bar_label(rects4, padding=3, fmt='%.1f%%', fontweight='bold')
ax4.set_ylim(0, 110)

plt.tight_layout()
plt.show()